# Data Preprocessing and Imputation Analysis Report
This notebook will show you my experiment about: how various missing value imputation techniques impact the accuracy of different machine learning classifiers on our Chronic Kidney Disease dataset.


# Library Imports and Setup

I begin =with importing the necessary data science libraries. Key components include IterativeImputer (aka MICE) and KNNImputer for advanced data recovery. I've also implemented a filter to suppress ConvergenceWarning messages to ensure a clean final report.

In [49]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
from sklearn.exceptions import ConvergenceWarning

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import accuracy_score

In [50]:
warnings.filterwarnings("ignore", category=ConvergenceWarning) # This line will suppress convergence warnings

# Load Dataset and Cleaning

I am applying precautionary cleaning to ensure the data is standardized. I replace potential markers like "?" with NaN and strip hidden whitespace from text columns. This ensures the machine learning models don't treat identical categories (like "ckd" and "ckd ") as different values.

In [51]:
df = pd.read_csv("../Datasets/MissingValues1/kidney_disease.csv")
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
print("30 rows of the Kidney Disease dataset:")
display(df.head(30).style.set_properties(**{'background-color': "#000000", 'border-color': 'black'}))

30 rows of the Kidney Disease dataset:


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.000000,80.000000,1.020000,1.000000,0.000000,nan,normal,notpresent,notpresent,121.000000,36.000000,1.200000,nan,nan,15.400000,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.000000,50.000000,1.020000,4.000000,0.000000,nan,normal,notpresent,notpresent,nan,18.000000,0.800000,nan,nan,11.300000,38,6000,nan,no,no,no,good,no,no,ckd
2,2,62.000000,80.000000,1.010000,2.000000,3.000000,normal,normal,notpresent,notpresent,423.000000,53.000000,1.800000,nan,nan,9.600000,31,7500,nan,no,yes,no,poor,no,yes,ckd
3,3,48.000000,70.000000,1.005000,4.000000,0.000000,normal,abnormal,present,notpresent,117.000000,56.000000,3.800000,111.000000,2.500000,11.200000,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.000000,80.000000,1.010000,2.000000,0.000000,normal,normal,notpresent,notpresent,106.000000,26.000000,1.400000,nan,nan,11.600000,35,7300,4.6,no,no,no,good,no,no,ckd
5,5,60.000000,90.000000,1.015000,3.000000,0.000000,nan,nan,notpresent,notpresent,74.000000,25.000000,1.100000,142.000000,3.200000,12.200000,39,7800,4.4,yes,yes,no,good,yes,no,ckd
6,6,68.000000,70.000000,1.010000,0.000000,0.000000,nan,normal,notpresent,notpresent,100.000000,54.000000,24.000000,104.000000,4.000000,12.400000,36,nan,nan,no,no,no,good,no,no,ckd
7,7,24.000000,nan,1.015000,2.000000,4.000000,normal,abnormal,notpresent,notpresent,410.000000,31.000000,1.100000,nan,nan,12.400000,44,6900,5,no,yes,no,good,yes,no,ckd
8,8,52.000000,100.000000,1.015000,3.000000,0.000000,normal,abnormal,present,notpresent,138.000000,60.000000,1.900000,nan,nan,10.800000,33,9600,4.0,yes,yes,no,good,no,yes,ckd
9,9,53.000000,90.000000,1.020000,2.000000,0.000000,abnormal,abnormal,present,notpresent,70.000000,107.000000,7.200000,114.000000,3.700000,9.500000,29,12100,3.7,yes,yes,no,poor,no,yes,ckd


# Let's Identify some Missing Data and do some Preparations

Before choosing an imputation strategy, I need to understand the "missingness" of the data. I've used a visual bar indicator to show that features like rbc (Red Blood Cell count) and rc (Red Cell count) have high levels of missing data, exceeding 30%. Then I eill map the target variable (classification) to numeric values and apply LabelEncoder to categorical features so they can be processed by our mathematical models.

In [52]:
missing_counts = df.isnull().sum().to_frame(name='Missing Count')
missing_counts['Percentage (%)'] = (df.isnull().sum() / len(df) * 100).round(2)
print(f"Total missing values in dataset: {df.isnull().sum().sum()}")
print("\nMissing values per column (Top 10 highest):")
display(missing_counts[missing_counts['Missing Count'] > 0]
        .sort_values(by='Missing Count', ascending=False)
        .head(10)
        .style.bar(subset=['Missing Count'], color='#d65f5f')
        .format({'Percentage (%)': '{:.2f}%'}))

Total missing values in dataset: 1009

Missing values per column (Top 10 highest):


,Missing Count,Percentage (%)
rbc,152,38.00%
rc,130,32.50%
wc,105,26.25%
pot,88,22.00%
sod,87,21.75%
pcv,70,17.50%
pc,65,16.25%
hemo,52,13.00%
su,49,12.25%
sg,47,11.75%


In [53]:
df["classification"] = df["classification"].map({"ckd": 1, "notckd": 0})

X = df.drop("classification", axis=1)
y = df["classification"]

In [54]:
for col in X.columns:
    try:
        X[col] = pd.to_numeric(X[col])
    except:
        pass

In [55]:
label_encoders = {}

for col in X.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Train/Test Split
To prevent data leakage, I will split the dataset before performing any imputation. This ensures that statistics used to fill missing values (like the mean or median) are derived only from the training set, mimicking real-world conditions where the future data is unknown.

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Here I Define Models and execute the nested loop

I have defined 12 unique imputation strategies and 4 classifiers. By using a nested loop, the code will automatically perform 48 independent experiments. This comprehensive approach reveals which combinations are most robust against missing data.

In [57]:
imputers = {
    "Mean Imputation": SimpleImputer(strategy="mean"),
    "Median Imputation": SimpleImputer(strategy="median"),
    "Most Frequent Imputation": SimpleImputer(strategy="most_frequent"),
    "Constant (0) Imputation": SimpleImputer(strategy="constant", fill_value=0),
    "KNN (k=3)": KNNImputer(n_neighbors=3),
    "KNN (k=5)": KNNImputer(n_neighbors=5),
    "KNN (k=10)": KNNImputer(n_neighbors=10),
    "MICE (BayesianRidge)": IterativeImputer(estimator=BayesianRidge(), random_state=42),
    "MICE (DecisionTree)": IterativeImputer(estimator=DecisionTreeRegressor(), random_state=42),
    "MICE (ExtraTrees)": IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=10), random_state=42),
    "MICE (Max Iter=5)": IterativeImputer(max_iter=5, random_state=42),
    "MICE (Iterative Imputer)": IterativeImputer(random_state=42, max_iter=20, tol=0.1)
}

In [58]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [59]:
results = []
for imputer_name, imputer in imputers.items():
    imputer.fit(X_train)
    X_train_imputed = imputer.transform(X_train)
    X_test_imputed = imputer.transform(X_test)
    
    for clf_name, clf in classifiers.items():
        clf.fit(X_train_imputed, y_train)
        y_pred = clf.predict(X_test_imputed)
        accuracy = accuracy_score(y_test, y_pred)
        results.append({
            "Imputation Algorithms": imputer_name,
            "Prediction Algorithms": clf_name,
            "Accuracy": accuracy
        })

# Final Results and Outcome Analysis

> The final matrix shows a side-by-side comparison of every experiment.

Key Takeaways:

- Tree-Based Dominance: Decision Tree and Random Forest achieved a perfect 1.0000 (100%) accuracy regardless of the imputation method. This indicates that the dataset contains very strong, clear features that these non-linear models can easily isolate.

- Classifier Sensitivity: Gaussian Naive Bayes was the most sensitive, consistently performing at 0.9750. Logistic Regression hovered around 0.9875, reaching perfection only with specific strategies like "Most Frequent" or "Constant" imputation.

- Imputation Impact: For this specific dataset, complex MICE or KNN methods did not significantly outperform simple Mean or Median imputation. This suggests the data points are highly distinct even with missing values.


In [60]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False)
pivot_df = results_df.pivot(index='Imputation Algorithms', 
                            columns='Prediction Algorithms', 
                            values='Accuracy')
print("\nAccuracy Comparison Matrix (Imputer vs. Classifier):")
display(pivot_df.style.background_gradient(cmap='Greens').format("{:.4f}"))



Accuracy Comparison Matrix (Imputer vs. Classifier):


Prediction Algorithms,Decision Tree,Gaussian Naive Bayes,Logistic Regression,Random Forest
Imputation Algorithms,,,,
Constant (0) Imputation,1.0000,0.9750,1.0000,1.0000
KNN (k=10),1.0000,0.9750,0.9875,1.0000
KNN (k=3),1.0000,0.9750,0.9875,1.0000
KNN (k=5),1.0000,0.9750,0.9875,1.0000
MICE (BayesianRidge),1.0000,0.9750,0.9875,1.0000
MICE (DecisionTree),1.0000,0.9750,0.9875,1.0000
MICE (ExtraTrees),1.0000,0.9750,0.9875,1.0000
MICE (Iterative Imputer),1.0000,0.9750,0.9875,1.0000
MICE (Max Iter=5),1.0000,0.9750,0.9875,1.0000
